# LeetCode 657 — Robot Return to Origin bằng Python

**Link đề:** https://leetcode.com/problems/robot-return-to-origin/
**Độ khó:** Easy · **Chủ đề:** String, Simulation

```python
class Solution:
    def judgeCircle(self, moves: str) -> bool:
        ...
```

## Mục tiêu học
1. Mô hình hóa bài toán bằng **tọa độ** `(x, y)` trên mặt phẳng.
2. Viết lời giải **mô phỏng** (simulation): đọc từng lệnh, cập nhật trạng thái.
3. Dùng `dict` ánh xạ ký tự → vector `(dx, dy)` thay cho chuỗi `if/elif` dài.
4. Nhận ra insight **đếm**: về gốc ⇔ `#U == #D` và `#L == #R` → lời giải 1 dòng với `str.count` / `Counter`.
5. Biết thêm cách dùng **số phức** để biểu diễn điểm 2D (mẹo Python).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Robot bắt đầu ở gốc tọa độ `(0, 0)`. Chuỗi `moves` cho biết từng bước đi: `'R'` phải, `'L'` trái, `'U'` lên, `'D'` xuống; mỗi bước dài như nhau. Hướng "mặt" của robot không quan trọng.

Trả `True` nếu sau khi đi hết, robot **quay về** `(0, 0)`.

**Ví dụ 1:** `moves = "UD"` → `True` (lên 1 rồi xuống 1).
**Ví dụ 2:** `moves = "LL"` → `False` (kết thúc ở `(-2, 0)`).

**Constraints:** `1 <= len(moves) <= 2·10^4`, chỉ gồm `'U' 'D' 'L' 'R'`.

Quy ước trục: `x` tăng sang phải, `y` tăng lên trên.

| Lệnh | `dx` | `dy` |
|---|---:|---:|
| `R` | +1 | 0 |
| `L` | −1 | 0 |
| `U` | 0 | +1 |
| `D` | 0 | −1 |

## 2. Nền tảng Python cần biết

### 2.1. Duyệt từng ký tự của chuỗi
`for ch in s:` lấy lần lượt từng ký tự. Chuỗi trong Python là **bất biến** (immutable) — ta chỉ đọc, không sửa.

In [1]:
moves = "URDL"
for i, ch in enumerate(moves):
    print(f"bước {i}: lệnh {ch}")

bước 0: lệnh U
bước 1: lệnh R
bước 2: lệnh D
bước 3: lệnh L


### 2.2. `if / elif / else` — chọn một nhánh
Python kiểm tra lần lượt từ trên xuống, gặp điều kiện đúng đầu tiên thì chạy nhánh đó và **bỏ qua** các nhánh còn lại.

In [2]:
def describe(ch: str) -> str:
    if ch == 'U':
        return "lên"
    elif ch == 'D':
        return "xuống"
    elif ch == 'L':
        return "trái"
    else:
        return "phải"

print([describe(c) for c in "UDLR"])

['lên', 'xuống', 'trái', 'phải']


### 2.3. `dict` ánh xạ ký tự → tuple `(dx, dy)`
Tuple `(a, b)` là bộ giá trị cố định. Gán tách `dx, dy = DIR[ch]` (tuple unpacking) lấy hai thành phần cùng lúc.

In [3]:
DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}
dx, dy = DIR['U']
print("U ->", dx, dy)
print("L ->", DIR['L'])

U -> 0 1
L -> (-1, 0)


### 2.3b. Gán đồng thời `x, y = x + dx, y + dy`
Python tính **toàn bộ vế phải trước** (tạo một tuple tạm), rồi mới gán cho các biến vế trái.
Nhờ vậy ta cập nhật hai tọa độ trong một dòng mà không sợ "dùng giá trị đã bị ghi đè".
Ví dụ kinh điển: đổi chỗ hai biến `a, b = b, a` không cần biến tạm.

In [4]:
x, y = 0, 0
dx, dy = 1, -1
x, y = x + dx, y + dy
print("sau một bước:", (x, y))

a, b2 = 3, 7
a, b2 = b2, a
print("đổi chỗ:", a, b2)

sau một bước: (1, -1)
đổi chỗ: 7 3


### 2.4. `str.count(sub)` và `collections.Counter`
- `"UUDL".count("U")` đếm số lần xuất hiện, duyệt cả chuỗi `O(n)`.
- `Counter("UUDL")` đếm tất cả ký tự trong **một** lượt.

In [5]:
from collections import Counter
s = "UUDLRR"
print("count U =", s.count("U"), "| count R =", s.count("R"))
c = Counter(s)
print(c, "| c['D'] =", c['D'], "| c['X'] =", c['X'])

count U = 2 | count R = 2
Counter({'U': 2, 'R': 2, 'D': 1, 'L': 1}) | c['D'] = 1 | c['X'] = 0


### 2.5. Số phức (complex) như điểm 2D
Python có sẵn số phức: `1j` là đơn vị ảo. Ta coi `x + y·j` là điểm `(x, y)`: cộng số phức = cộng vector.

In [6]:
p = 0 + 0j
p += 1      # sang phải
p += 1j     # lên
p += 1j     # lên
print(p, "| x =", p.real, "| y =", p.imag)

(1+2j) | x = 1.0 | y = 2.0


## 3. Cách trực giác — mô phỏng với `if/elif`

Giữ `x, y = 0, 0`. Đọc từng lệnh và cộng/trừ. Cuối cùng kiểm tra `x == 0 and y == 0`.

Đây đã là lời giải `O(n)`; không có brute force "tệ" hơn tự nhiên. Ta vẫn đi từ cách này (dễ nghĩ nhất) đến các cách gọn hơn.

In [7]:
def judge_circle_simulate(moves: str) -> bool:
    x, y = 0, 0
    for ch in moves:
        if ch == 'U':
            y += 1
        elif ch == 'D':
            y -= 1
        elif ch == 'L':
            x -= 1
        elif ch == 'R':
            x += 1
    return x == 0 and y == 0

print(judge_circle_simulate("UD"))     # True
print(judge_circle_simulate("LL"))     # False
print(judge_circle_simulate("RRDDLLUU"))  # True

True
False
True


### 3.1. Một cách "ngây thơ" hơn: lưu cả đường đi
Có bạn lưu toàn bộ các điểm đã đi vào list rồi xem điểm cuối. Đúng, nhưng tốn `O(n)` bộ nhớ không cần thiết — ta chỉ cần **vị trí hiện tại**.

In [8]:
def judge_circle_path(moves: str) -> bool:
    DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}
    path = [(0, 0)]
    for ch in moves:
        dx, dy = DIR[ch]
        x, y = path[-1]
        path.append((x + dx, y + dy))
    print("đường đi:", path)
    return path[-1] == (0, 0)

judge_circle_path("URDL")

đường đi: [(0, 0), (0, 1), (1, 1), (1, 0), (0, 0)]


True

## 4. Nút thắt và insight — thứ tự không quan trọng!

Phép cộng vector có **tính giao hoán**: đi `"UR"` hay `"RU"` đều tới `(1, 1)`.
Vậy vị trí cuối chỉ phụ thuộc vào **số lượng** mỗi lệnh:

```text
x_cuối = #R − #L
y_cuối = #U − #D
```

Về gốc ⇔ `#R == #L` **và** `#U == #D`.

Đây là insight biến bài mô phỏng thành bài **đếm**.

In [9]:
for s in ["UR", "RU", "UUDLRL", "LDRRUL"]:
    print(f"{s:<8} -> x = {s.count('R') - s.count('L')}, y = {s.count('U') - s.count('D')}, "
          f"mô phỏng: {judge_circle_simulate(s)}")

UR       -> x = 1, y = 1, mô phỏng: False
RU       -> x = 1, y = 1, mô phỏng: False
UUDLRL   -> x = -1, y = 1, mô phỏng: False
LDRRUL   -> x = 0, y = 0, mô phỏng: True


## 5. Lời giải chính — đếm (giảng từng đoạn)

**Đoạn 1:** đếm 4 loại lệnh bằng `Counter(moves)` (một lượt duyệt).
**Đoạn 2:** so `c['U'] == c['D']`.
**Đoạn 3:** so `c['L'] == c['R']`.
**Đoạn 4:** trả `and` của hai điều kiện.

Lợi ích của `Counter`: lệnh không xuất hiện → đếm là `0`, không gây `KeyError`.

In [10]:
def judge_circle_count(moves: str) -> bool:
    c = Counter(moves)                 # đoạn 1
    vertical_ok = c['U'] == c['D']     # đoạn 2
    horizontal_ok = c['L'] == c['R']   # đoạn 3
    return vertical_ok and horizontal_ok   # đoạn 4

for s in ["UD", "LL", "RRDDLLUU", "U"]:
    print(s, "->", judge_circle_count(s))

UD -> True
LL -> False
RRDDLLUU -> True
U -> False


## 6. TRACE từng bước

### 6.1. Trace mô phỏng với `moves = "URDDLU"`

| Bước | Lệnh | `(dx, dy)` | `(x, y)` sau |
|---:|---|---|---|
| 0 | – | – | `(0, 0)` |
| 1 | U | (0, +1) | `(0, 1)` |
| 2 | R | (+1, 0) | `(1, 1)` |
| 3 | D | (0, −1) | `(1, 0)` |
| 4 | D | (0, −1) | `(1, −1)` |
| 5 | L | (−1, 0) | `(0, −1)` |
| 6 | U | (0, +1) | `(0, 0)` ✔ |

In [11]:
def trace_judge_circle(moves: str) -> bool:
    DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}
    x = y = 0
    print(f"bắt đầu: ({x}, {y})")
    for i, ch in enumerate(moves, start=1):
        dx, dy = DIR[ch]
        x, y = x + dx, y + dy
        print(f"bước {i}: lệnh {ch} (dx={dx:+d}, dy={dy:+d}) -> ({x}, {y})")
    ans = (x, y) == (0, 0)
    print("về gốc?", ans)
    return ans

trace_judge_circle("URDDLU")

bắt đầu: (0, 0)
bước 1: lệnh U (dx=+0, dy=+1) -> (0, 1)
bước 2: lệnh R (dx=+1, dy=+0) -> (1, 1)
bước 3: lệnh D (dx=+0, dy=-1) -> (1, 0)
bước 4: lệnh D (dx=+0, dy=-1) -> (1, -1)
bước 5: lệnh L (dx=-1, dy=+0) -> (0, -1)
bước 6: lệnh U (dx=+0, dy=+1) -> (0, 0)
về gốc? True


True

### 6.2. Trace cách đếm với `moves = "UURDL"`
`#U=2, #D=1` → khác nhau → `False` ngay, dù `#L == #R`.

In [12]:
def trace_count(moves: str) -> bool:
    c = Counter(moves)
    print("đếm:", {k: c[k] for k in "UDLR"})
    print(f"U == D ? {c['U']} == {c['D']} -> {c['U'] == c['D']}")
    print(f"L == R ? {c['L']} == {c['R']} -> {c['L'] == c['R']}")
    return c['U'] == c['D'] and c['L'] == c['R']

print("kết quả:", trace_count("UURDL"))

đếm: {'U': 2, 'D': 1, 'L': 1, 'R': 1}
U == D ? 2 == 1 -> False
L == R ? 1 == 1 -> True
kết quả: False


## 7. Code nộp LeetCode

### 7.1. Cách 1 — Đếm bằng `str.count` (ngắn, nhanh vì `count` chạy bằng C)

In [13]:
class Solution:
    def judgeCircle(self, moves: str) -> bool:
        return moves.count('U') == moves.count('D') and moves.count('L') == moves.count('R')

print(Solution().judgeCircle("UD"), Solution().judgeCircle("LL"))

True False


### 7.2. Cách 2 — Mô phỏng tọa độ với `dict` hướng

In [14]:
class SolutionSimulate:
    def judgeCircle(self, moves: str) -> bool:
        DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}
        x = y = 0
        for ch in moves:
            dx, dy = DIR[ch]
            x += dx
            y += dy
        return x == 0 and y == 0

print(SolutionSimulate().judgeCircle("UD"), SolutionSimulate().judgeCircle("LL"))

True False


### 7.3. Cách 3 — `Counter`

In [15]:
class SolutionCounter:
    def judgeCircle(self, moves: str) -> bool:
        c = Counter(moves)
        return c['U'] == c['D'] and c['L'] == c['R']

print(SolutionCounter().judgeCircle("RRDDLLUU"))

True


### 7.4. Cách 4 — Số phức (mẹo Python)
`sum(...)` với `start=0j` cộng dồn các số phức. `1` = phải, `-1` = trái, `1j` = lên, `-1j` = xuống.

In [16]:
class SolutionComplex:
    def judgeCircle(self, moves: str) -> bool:
        step = {'R': 1, 'L': -1, 'U': 1j, 'D': -1j}
        return sum((step[ch] for ch in moves), 0j) == 0

print(SolutionComplex().judgeCircle("UD"), SolutionComplex().judgeCircle("LL"))

True False


## 8. Vì sao đúng?

**Mô phỏng — bất biến:** sau khi đọc `i` lệnh đầu, `(x, y)` là vị trí thực của robot sau `i` bước (mỗi bước cộng đúng vector của lệnh). Sau `n` lệnh, `(x, y)` là vị trí cuối → so với `(0, 0)`.

**Đếm:** vị trí cuối = tổng các vector = `#R·(1,0) + #L·(−1,0) + #U·(0,1) + #D·(0,−1)` = `(#R − #L, #U − #D)`. Hai thành phần bằng 0 ⇔ `#R = #L` và `#U = #D`. Phép cộng giao hoán nên thứ tự không ảnh hưởng.

**Hệ quả hay:** nếu `len(moves)` **lẻ** thì chắc chắn `False` (vì `#R+#L+#U+#D` = `2#R + 2#U` phải chẵn khi về gốc).

In [17]:
for s in ["U", "UDL", "UDLRU"]:
    print(s, "độ dài lẻ ->", Solution().judgeCircle(s))

U độ dài lẻ -> False
UDL độ dài lẻ -> False
UDLRU độ dài lẻ -> False


### 8.1. Tối ưu nhỏ: thoát sớm khi độ dài lẻ
Từ hệ quả trên, có thể thêm `if len(moves) % 2: return False` trước khi đếm. Không đổi Big-O (`len` của chuỗi là `O(1)`), nhưng một nửa số input ngẫu nhiên được trả lời ngay. Đây là kiểu **lập luận chẵn lẻ (parity)** rất hay gặp: tìm một đại lượng mà mọi đáp án "đúng" phải thỏa, để loại nhanh các ca chắc chắn sai.

In [18]:
def judge_circle_parity(moves: str) -> bool:
    if len(moves) % 2 == 1:          # lẻ -> không thể về gốc
        return False
    return moves.count('U') == moves.count('D') and moves.count('L') == moves.count('R')

for s in ["UD", "UDL", "LRUD", "LLRRU"]:
    print(f"{s:<6} len={len(s)} -> {judge_circle_parity(s)}")

UD     len=2 -> True
UDL    len=3 -> False
LRUD   len=4 -> True
LLRRU  len=5 -> False


## 9. Độ phức tạp và so sánh

Gọi `n = len(moves)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Mô phỏng `if/elif` | `O(n)` | `O(1)` | Dễ nghĩ nhất |
| Lưu cả đường đi | `O(n)` | `O(n)` | Lãng phí bộ nhớ |
| Mô phỏng + `dict` | `O(n)` | `O(1)` | Dễ mở rộng thêm hướng |
| `str.count` × 4 | `O(n)` (4 lượt) | `O(1)` | Ngắn, nhanh thực tế |
| `Counter` | `O(n)` | `O(1)` (≤ 4 key) | Một lượt |
| Số phức | `O(n)` | `O(1)` | Mẹo, ít phổ biến |

In [19]:
import time, random
rng = random.Random(657)
big = "".join(rng.choice("UDLR") for _ in range(200_000))
for name, f in [("count", Solution().judgeCircle), ("simulate", SolutionSimulate().judgeCircle),
                ("Counter", SolutionCounter().judgeCircle), ("complex", SolutionComplex().judgeCircle)]:
    t = time.perf_counter(); r = f(big)
    print(f"{name:>9}: {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

    count: False  (1.4 ms)


 simulate: False  (14.0 ms)


  Counter: False  (13.2 ms)


  complex: False  (18.2 ms)


## 10. Bộ test

1. Ví dụ đề: `"UD"`, `"LL"`.
2. Một ký tự (luôn `False`).
3. Về gốc bằng vòng tròn: `"URDL"`, `"RRDDLLUU"`.
4. Cân một trục, lệch trục kia: `"UDL"`, `"LRU"`.
5. Ngẫu nhiên: so mọi cách với mô phỏng `if/elif`.

In [20]:
cases = [
    ("UD", True), ("LL", False), ("U", False), ("R", False),
    ("URDL", True), ("RRDDLLUU", True), ("UDL", False), ("LRU", False),
    ("UUDDLRLR", True), ("LDRRLRUULR", False),
]
solvers = [Solution().judgeCircle, SolutionSimulate().judgeCircle, SolutionCounter().judgeCircle,
           SolutionComplex().judgeCircle, judge_circle_simulate, judge_circle_count]
for s, expected in cases:
    for f in solvers:
        assert f(s) == expected, (f, s)
    print(f"{s:<12} -> {expected} PASS")

for _ in range(500):
    s = "".join(rng.choice("UDLR") for _ in range(rng.randint(1, 12)))
    exp = judge_circle_simulate(s)
    assert all(f(s) == exp for f in solvers)
print("500 chuỗi ngẫu nhiên: mọi cách khớp nhau PASS")

UD           -> True PASS
LL           -> False PASS
U            -> False PASS
R            -> False PASS
URDL         -> True PASS
RRDDLLUU     -> True PASS
UDL          -> False PASS
LRU          -> False PASS
UUDDLRLR     -> True PASS
LDRRLRUULR   -> False PASS
500 chuỗi ngẫu nhiên: mọi cách khớp nhau PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra **tổng** `x + y == 0`
`"UL"` cho `x = -1, y = 1`, tổng = 0 nhưng robot không ở gốc!

In [21]:
def wrong_sum_xy(moves):
    x = moves.count('R') - moves.count('L')
    y = moves.count('U') - moves.count('D')
    return x + y == 0                       # SAI

print("sai:", wrong_sum_xy("UL"), "| đúng:", Solution().judgeCircle("UL"))

sai: True | đúng: False


### Lỗi 2: Ghép nhầm cặp đối lập
So `#U == #L` và `#D == #R` — sai ngữ nghĩa: U đối lập với **D**, L đối lập với **R**.

In [22]:
def wrong_pairs(moves):
    return moves.count('U') == moves.count('L') and moves.count('D') == moves.count('R')   # SAI

print("sai:", wrong_pairs("UD"), "| đúng:", Solution().judgeCircle("UD"))
print("sai:", wrong_pairs("UL"), "| đúng:", Solution().judgeCircle("UL"))

sai: False | đúng: True
sai: True | đúng: False


### Lỗi 3: Dùng `or` thay vì `and`
`#U == #D or #L == #R` → `"LL"` có `#U == #D == 0` → trả `True` sai.

### Lỗi 4: Dùng `dict` thường rồi truy cập key không có
`d = {}; for ch in moves: d[ch] = d.get(ch, 0) + 1` rồi `d['U'] == d['D']` → `KeyError` khi chuỗi không có `'U'`. Dùng `d.get('U', 0)` hoặc `Counter`.

In [23]:
def wrong_or(moves):
    return moves.count('U') == moves.count('D') or moves.count('L') == moves.count('R')   # SAI

print("sai:", wrong_or("LL"), "| đúng:", Solution().judgeCircle("LL"))

d = {}
for ch in "LR":
    d[ch] = d.get(ch, 0) + 1
try:
    print(d['U'] == d['D'])
except KeyError as e:
    print("KeyError:", e, "-> dùng d.get('U', 0) hoặc Counter")

sai: True | đúng: False
KeyError: 'U' -> dùng d.get('U', 0) hoặc Counter


## 12. Mẹo nhận dạng pattern

- **"Đi theo chuỗi lệnh, hỏi vị trí cuối"** → mô phỏng tọa độ `(x, y)` với bảng hướng `dict`.
- Nếu chỉ hỏi **vị trí cuối** (không hỏi đường đi, không có vật cản) → thứ tự không quan trọng → **đếm**.
- Nếu hỏi "có **đi qua** điểm nào hai lần?" (LeetCode 1496 Path Crossing) → phải mô phỏng và lưu `set` các điểm đã thăm.
- Nếu robot có **hướng mặt** và lệnh "quay trái/phải" (LeetCode 1041) → mô phỏng với chỉ số hướng `d = (d + 1) % 4`.

## 13. Biến thể

### 13.1. Path Crossing (LeetCode 1496): đường đi có tự cắt không?

In [24]:
def is_path_crossing(path: str) -> bool:
    DIR = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
    x = y = 0
    visited = {(0, 0)}
    for ch in path:
        dx, dy = DIR[ch]
        x, y = x + dx, y + dy
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False

print(is_path_crossing("NES"), is_path_crossing("NESWW"))   # False True

False True


### 13.2. Robot Bounded In Circle (LeetCode 1041)
Lệnh `G` (đi thẳng), `L`/`R` (quay 90°). Lặp chuỗi mãi; robot bị giới hạn trong vòng tròn ⇔ sau **một** lượt robot về gốc **hoặc** không còn hướng Bắc.

In [25]:
def is_robot_bounded(instructions: str) -> bool:
    dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]   # Bắc, Đông, Nam, Tây (theo chiều kim đồng hồ)
    x = y = 0
    d = 0
    for ch in instructions:
        if ch == 'G':
            x += dirs[d][0]; y += dirs[d][1]
        elif ch == 'R':
            d = (d + 1) % 4
        else:
            d = (d + 3) % 4
    return (x, y) == (0, 0) or d != 0

print(is_robot_bounded("GGLLGG"), is_robot_bounded("GG"), is_robot_bounded("GL"))  # True False True

True False True


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** robot đi `"RRULDDLU"` kết thúc ở đâu? Có về gốc không?

**Bài 2 — Đoán output:**
```python
print("LRUDUD".count("U"), Counter("LRUDUD")["X"], sum([1j, -1, 1, -1j], 0j))
```

**Bài 3 — TODO:** trả về **tọa độ cuối** `(x, y)` của robot.

In [26]:
def exercise_final_position(moves: str) -> tuple:
    # TODO: mô phỏng với dict hướng, trả (x, y)
    pass

# assert exercise_final_position("UD") == (0, 0)
# assert exercise_final_position("LLU") == (-2, 1)

**Bài 4 — TODO:** trả về **số lần** robot đi qua gốc `(0, 0)` **sau khi xuất phát** (không tính vị trí bắt đầu).

In [27]:
def exercise_count_returns(moves: str) -> int:
    # TODO: mô phỏng; sau mỗi bước nếu (x, y) == (0, 0) thì tăng biến đếm
    pass

# assert exercise_count_returns("UDUD") == 2
# assert exercise_count_returns("LL") == 0

**Bài 5 — TODO:** tìm **khoảng cách Manhattan lớn nhất** `|x| + |y|` mà robot từng đạt được trong hành trình.

In [28]:
def exercise_max_manhattan(moves: str) -> int:
    # TODO: mô phỏng, cập nhật best = max(best, abs(x) + abs(y))
    pass

# assert exercise_max_manhattan("UURDDD") == 3
# assert exercise_max_manhattan("UD") == 1

**Bài 6 — Mini challenge:** cho `moves` **chưa** về gốc, trả về **số lệnh tối thiểu cần thêm vào cuối** để robot về gốc.

In [29]:
def exercise_min_extra_moves(moves: str) -> int:
    # TODO: tính (x, y) cuối; mỗi lệnh thêm giảm |x| hoặc |y| đi 1
    pass

# assert exercise_min_extra_moves("UD") == 0
# assert exercise_min_extra_moves("LLU") == 3

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đếm từng loại | `#R=2, #L=2` | `#U=2, #D=2` → gốc |
| 2 | `count` đếm chuỗi con | `Counter` trả 0 cho key thiếu | tổng số phức: `1j - 1 + 1 - 1j` |
| 3 | Copy phần mô phỏng | `DIR[ch]` | `return (x, y)` |
| 4 | Kiểm tra **sau** mỗi bước | `if x == 0 and y == 0: cnt += 1` | không kiểm tra trước vòng lặp |
| 5 | Theo dõi max trong vòng lặp | `abs()` giá trị tuyệt đối | `best = 0` ban đầu |
| 6 | Không cần mô phỏng thêm | Khoảng cách Manhattan tới gốc | `abs(x) + abs(y)` |

## 16. Đáp án tham khảo

**Bài 1:** `#R = 2, #L = 2, #U = 2, #D = 2` → `(0, 0)`, về gốc.

**Bài 2:** `2 0 0j`.

In [30]:
print("Bài 1:", Solution().judgeCircle("RRULDDLU"))
print("Bài 2:", "LRUDUD".count("U"), Counter("LRUDUD")["X"], sum([1j, -1, 1, -1j], 0j))

Bài 1: True
Bài 2: 2 0 0j


In [31]:
DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}

def answer_final_position(moves: str) -> tuple:
    x = y = 0
    for ch in moves:
        dx, dy = DIR[ch]
        x += dx; y += dy
    return (x, y)


def answer_count_returns(moves: str) -> int:
    x = y = cnt = 0
    for ch in moves:
        dx, dy = DIR[ch]
        x += dx; y += dy
        if x == 0 and y == 0:
            cnt += 1
    return cnt


def answer_max_manhattan(moves: str) -> int:
    x = y = best = 0
    for ch in moves:
        dx, dy = DIR[ch]
        x += dx; y += dy
        best = max(best, abs(x) + abs(y))
    return best


def answer_min_extra_moves(moves: str) -> int:
    x, y = answer_final_position(moves)
    return abs(x) + abs(y)


assert answer_final_position("UD") == (0, 0)
assert answer_final_position("LLU") == (-2, 1)
assert answer_count_returns("UDUD") == 2
assert answer_count_returns("LL") == 0
assert answer_max_manhattan("UURDDD") == 3
assert answer_max_manhattan("UD") == 1
assert answer_min_extra_moves("UD") == 0
assert answer_min_extra_moves("LLU") == 3
for _ in range(200):
    s = "".join(rng.choice("UDLR") for _ in range(rng.randint(1, 15)))
    assert (answer_min_extra_moves(s) == 0) == Solution().judgeCircle(s)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def judgeCircle(self, moves: str) -> bool:
        return moves.count('U') == moves.count('D') and moves.count('L') == moves.count('R')
```

Template mô phỏng tổng quát (dùng cho các bài robot khác):
```python
DIR = {'R': (1, 0), 'L': (-1, 0), 'U': (0, 1), 'D': (0, -1)}
x = y = 0
for ch in moves:
    dx, dy = DIR[ch]
    x += dx; y += dy
```

**Checklist:**
- [ ] Ghép đúng cặp đối lập: U↔D, L↔R.
- [ ] Dùng `and` (cả hai trục đều về 0), không dùng `or`, không dùng `x + y == 0`.
- [ ] Không `KeyError` khi một lệnh không xuất hiện.
- [ ] Time `O(n)`, Space `O(1)`.